# Assignment 6 : Spark Architecture & Data Processing using PySpark

## Objective
This notebook demonstrates the implementation of Spark Architecture concepts and efficient data processing using PySpark.

### Topics Covered
- Spark Architecture
- Spark Execution Modes
- Lazy Evaluation
- DAG (Directed Acyclic Graph)
- Reading CSV Files
- Schema Handling
- Data Exploration
- DataFrame Transformations
- Performance Optimization
- ETL Pipeline

#Install PySpark

In this step, we install the PySpark library in Google Colab. PySpark provides the Python API for Apache Spark, allowing us to perform distributed data processing.

In [1]:
# Install PySpark

!pip install pyspark -q

#Import Required Libraries

Import all the necessary PySpark libraries that will be used throughout this assignment.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

#  Create a Spark Session

A Spark Session is the entry point for every Spark application. It allows us to create DataFrames, execute SQL queries, and interact with the Spark cluster.

In [3]:
spark = SparkSession.builder \
    .appName("Hotel Booking Demand Assignment") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.3


# Understand Spark Architecture

Apache Spark follows a Master-Worker Architecture.

### Components:
- Driver Program
- Cluster Manager
- Executors

The Driver creates the execution plan, the Cluster Manager allocates resources, and Executors perform the actual computations.

# Upload the Dataset

Upload the downloaded **hotel_bookings.csv** file from your local system to Google Colab.

In [4]:
from google.colab import files

uploaded = files.upload()

Saving hotel_bookings.csv to hotel_bookings.csv


# Read the CSV Dataset

Read the Hotel Booking Demand dataset into a Spark DataFrame with automatic schema detection.

In [5]:
df = spark.read.csv(
    "hotel_bookings.csv",
    header=True,
    inferSchema=True
)

#Display the Dataset

Display the first five records to verify that the dataset has been loaded correctly.

In [6]:
df.show(5, truncate=False)

+------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+----+---------------------------+-------------------------+------------------+-----------------------+
|hotel       |is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type|adr |required_car_parking_spaces|total_

# Display the Schema

Print the schema to understand the structure of the dataset, including column names and data types.

In [7]:
df.printSchema()

root
 |-- hotel: string (nullable = true)
 |-- is_canceled: integer (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- arrival_date_year: integer (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: integer (nullable = true)
 |-- arrival_date_day_of_month: integer (nullable = true)
 |-- stays_in_weekend_nights: integer (nullable = true)
 |-- stays_in_week_nights: integer (nullable = true)
 |-- adults: integer (nullable = true)
 |-- children: string (nullable = true)
 |-- babies: integer (nullable = true)
 |-- meal: string (nullable = true)
 |-- country: string (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- is_repeated_guest: integer (nullable = true)
 |-- previous_cancellations: integer (nullable = true)
 |-- previous_bookings_not_canceled: integer (nullable = true)
 |-- reserved_room_type: string (nullable = true)
 |-- assigned_room_type: string (nullab

#  Count Rows and Columns

Check the total number of rows and columns present in the dataset.

In [8]:
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

Total Rows: 119390
Total Columns: 32


# Display Column Names

Print all column names available in the dataset.

In [9]:
print(df.columns)

['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']


# Generate Summary Statistics

Display statistical information such as count, mean, minimum, and maximum values for the dataset.

In [10]:
df.describe().show()

+-------+------------+-------------------+------------------+------------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------------------+-------------------+--------------------+---------+-------+--------------+--------------------+-------------------+----------------------+------------------------------+------------------+------------------+-------------------+------------+------------------+------------------+--------------------+---------------+------------------+---------------------------+-------------------------+------------------+
|summary|       hotel|        is_canceled|         lead_time| arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|            adults|           children|              babies|     meal|country|market_segment|distribution_channel|  is_repeated_guest|previous_cancellations|previous_bookings_not_canc

# Check Data Types

Display the data type of each column in the DataFrame.

In [11]:
df.dtypes

[('hotel', 'string'),
 ('is_canceled', 'int'),
 ('lead_time', 'int'),
 ('arrival_date_year', 'int'),
 ('arrival_date_month', 'string'),
 ('arrival_date_week_number', 'int'),
 ('arrival_date_day_of_month', 'int'),
 ('stays_in_weekend_nights', 'int'),
 ('stays_in_week_nights', 'int'),
 ('adults', 'int'),
 ('children', 'string'),
 ('babies', 'int'),
 ('meal', 'string'),
 ('country', 'string'),
 ('market_segment', 'string'),
 ('distribution_channel', 'string'),
 ('is_repeated_guest', 'int'),
 ('previous_cancellations', 'int'),
 ('previous_bookings_not_canceled', 'int'),
 ('reserved_room_type', 'string'),
 ('assigned_room_type', 'string'),
 ('booking_changes', 'int'),
 ('deposit_type', 'string'),
 ('agent', 'string'),
 ('company', 'string'),
 ('days_in_waiting_list', 'int'),
 ('customer_type', 'string'),
 ('adr', 'double'),
 ('required_car_parking_spaces', 'int'),
 ('total_of_special_requests', 'int'),
 ('reservation_status', 'string'),
 ('reservation_status_date', 'date')]

#Understand Lazy Evaluation

Spark uses Lazy Evaluation, which means transformations are not executed immediately. Instead, Spark builds a logical execution plan (DAG). The execution starts only when an action such as `show()`, `count()`, or `collect()` is called.

# Create a Transformation

Apply a filter transformation to select bookings with a lead time greater than 100 days. No computation occurs at this stage.

In [12]:
filtered_df = df.filter(col("lead_time") > 100)

print("Transformation Created Successfully")

Transformation Created Successfully


# Trigger an Action

The `show()` function is an action that forces Spark to execute the pending transformations and display the results.

In [13]:
filtered_df.show(10)

+------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+---------------+-----+---------------------------+-------------------------+------------------+-----------------------+
|       hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|  customer_type|  adr|required_car_parking_spaces|

# Display the DAG (Execution Plan)

Use the `explain(True)` function to display the logical and physical execution plans generated by Spark.

In [14]:
filtered_df.explain(True)

== Parsed Logical Plan ==
'Filter '`>`('lead_time, 100)
+- Relation [hotel#17,is_canceled#18,lead_time#19,arrival_date_year#20,arrival_date_month#21,arrival_date_week_number#22,arrival_date_day_of_month#23,stays_in_weekend_nights#24,stays_in_week_nights#25,adults#26,children#27,babies#28,meal#29,country#30,market_segment#31,distribution_channel#32,is_repeated_guest#33,previous_cancellations#34,previous_bookings_not_canceled#35,reserved_room_type#36,assigned_room_type#37,booking_changes#38,deposit_type#39,agent#40,company#41,... 7 more fields] csv

== Analyzed Logical Plan ==
hotel: string, is_canceled: int, lead_time: int, arrival_date_year: int, arrival_date_month: string, arrival_date_week_number: int, arrival_date_day_of_month: int, stays_in_weekend_nights: int, stays_in_week_nights: int, adults: int, children: string, babies: int, meal: string, country: string, market_segment: string, distribution_channel: string, is_repeated_guest: int, previous_cancellations: int, previous_bookin

# DataFrame Transformations and Data Cleaning

In this section, we will perform common DataFrame operations such as selecting columns, filtering rows, renaming columns, changing data types, adding new columns, and handling missing values.

#Select Required Columns

Select only the columns that are useful for analysis. This improves readability and reduces unnecessary data processing.

In [15]:
# Select important columns

selected_df = df.select(
    "hotel",
    "lead_time",
    "arrival_date_year",
    "country",
    "adults",
    "children",
    "babies",
    "adr"
)

selected_df.show(10)

+------------+---------+-----------------+-------+------+--------+------+-----+
|       hotel|lead_time|arrival_date_year|country|adults|children|babies|  adr|
+------------+---------+-----------------+-------+------+--------+------+-----+
|Resort Hotel|      342|             2015|    PRT|     2|       0|     0|  0.0|
|Resort Hotel|      737|             2015|    PRT|     2|       0|     0|  0.0|
|Resort Hotel|        7|             2015|    GBR|     1|       0|     0| 75.0|
|Resort Hotel|       13|             2015|    GBR|     1|       0|     0| 75.0|
|Resort Hotel|       14|             2015|    GBR|     2|       0|     0| 98.0|
|Resort Hotel|       14|             2015|    GBR|     2|       0|     0| 98.0|
|Resort Hotel|        0|             2015|    PRT|     2|       0|     0|107.0|
|Resort Hotel|        9|             2015|    PRT|     2|       0|     0|103.0|
|Resort Hotel|       85|             2015|    PRT|     2|       0|     0| 82.0|
|Resort Hotel|       75|             201

#Filter Hotel Bookings

Filter the dataset to display only bookings for Resort Hotels.

In [16]:
# Filter Resort Hotel bookings

resort_df = df.filter(col("hotel") == "Resort Hotel")

resort_df.show(10)

+------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+-----+---------------------------+-------------------------+------------------+-----------------------+
|       hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type|  adr|required_car_parking_spaces|tota

# Apply Multiple Filter Conditions

Display bookings where:
- Lead time is greater than 100 days.
- Number of adults is at least 2.

In [17]:
filtered_df = df.filter(
    (col("lead_time") > 100) &
    (col("adults") >= 2)
)

filtered_df.show(10)

+------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+---------------+------+---------------------------+-------------------------+------------------+-----------------------+
|       hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|  customer_type|   adr|required_car_parking_space

# Rename a Column

Rename the 'adr' column to 'Average_Daily_Rate' to make it more meaningful.

In [18]:
renamed_df = df.withColumnRenamed(
    "adr",
    "Average_Daily_Rate"
)

renamed_df.show(5)

+------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+------------------+---------------------------+-------------------------+------------------+-----------------------+
|       hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type|Average_Daily_Rate|requir

# Display Updated Schema

Verify that the column has been renamed successfully.

In [19]:
renamed_df.printSchema()

root
 |-- hotel: string (nullable = true)
 |-- is_canceled: integer (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- arrival_date_year: integer (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: integer (nullable = true)
 |-- arrival_date_day_of_month: integer (nullable = true)
 |-- stays_in_weekend_nights: integer (nullable = true)
 |-- stays_in_week_nights: integer (nullable = true)
 |-- adults: integer (nullable = true)
 |-- children: string (nullable = true)
 |-- babies: integer (nullable = true)
 |-- meal: string (nullable = true)
 |-- country: string (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- is_repeated_guest: integer (nullable = true)
 |-- previous_cancellations: integer (nullable = true)
 |-- previous_bookings_not_canceled: integer (nullable = true)
 |-- reserved_room_type: string (nullable = true)
 |-- assigned_room_type: string (nullab

# Cast a Column to a Different Data Type

Convert the 'children' column to Integer type.

In [20]:
cast_df = df.withColumn(
    "children",
    col("children").cast("Integer")
)

cast_df.printSchema()

root
 |-- hotel: string (nullable = true)
 |-- is_canceled: integer (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- arrival_date_year: integer (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: integer (nullable = true)
 |-- arrival_date_day_of_month: integer (nullable = true)
 |-- stays_in_weekend_nights: integer (nullable = true)
 |-- stays_in_week_nights: integer (nullable = true)
 |-- adults: integer (nullable = true)
 |-- children: integer (nullable = true)
 |-- babies: integer (nullable = true)
 |-- meal: string (nullable = true)
 |-- country: string (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- is_repeated_guest: integer (nullable = true)
 |-- previous_cancellations: integer (nullable = true)
 |-- previous_bookings_not_canceled: integer (nullable = true)
 |-- reserved_room_type: string (nullable = true)
 |-- assigned_room_type: string (nulla

# Add a New Column

Create a new column named 'Total_Guests' by adding adults, children, and babies.

In [21]:
new_df = df.withColumn(
    "Total_Guests",
    col("adults") + col("children") + col("babies")
)

new_df.select(
    "adults",
    "children",
    "babies",
    "Total_Guests"
).show(10)

+------+--------+------+------------+
|adults|children|babies|Total_Guests|
+------+--------+------+------------+
|     2|       0|     0|           2|
|     2|       0|     0|           2|
|     1|       0|     0|           1|
|     1|       0|     0|           1|
|     2|       0|     0|           2|
|     2|       0|     0|           2|
|     2|       0|     0|           2|
|     2|       0|     0|           2|
|     2|       0|     0|           2|
|     2|       0|     0|           2|
+------+--------+------+------------+
only showing top 10 rows


# Add a Discounted Room Price

Create a new column that calculates the room price after applying a 10% discount.

In [22]:
price_df = df.withColumn(
    "Discounted_ADR",
    round(col("adr") * 0.90, 2)
)

price_df.select(
    "adr",
    "Discounted_ADR"
).show(10)

+-----+--------------+
|  adr|Discounted_ADR|
+-----+--------------+
|  0.0|           0.0|
|  0.0|           0.0|
| 75.0|          67.5|
| 75.0|          67.5|
| 98.0|          88.2|
| 98.0|          88.2|
|107.0|          96.3|
|103.0|          92.7|
| 82.0|          73.8|
|105.5|         94.95|
+-----+--------------+
only showing top 10 rows


# Display Missing Values

Count the number of null values in each column.

In [23]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+-----+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+---+---------------------------+-------------------------+------------------+-----------------------+
|hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type|adr|required_car_parking_spaces|total_of_special_reque

# Handle Missing Values Using fillna()

Replace missing values with suitable defaults.

In [24]:
filled_df = df.fillna({

    "children": 0,

    "country": "Unknown",

    "agent": 0,

    "company": 0

})

filled_df.show(5)

+------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+----+---------------------------+-------------------------+------------------+-----------------------+
|       hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type| adr|required_car_parking_spaces|total_

# Remove Rows with Missing Values

Drop rows that contain null values.

In [25]:
drop_df = df.dropna()

print("Original Rows :", df.count())

print("Rows After Drop :", drop_df.count())

Original Rows : 119390
Rows After Drop : 119390


# Remove Duplicate Records

Remove duplicate rows from the dataset.

In [26]:
duplicate_removed = df.dropDuplicates()

print("Original Records :", df.count())

print("After Removing Duplicates :", duplicate_removed.count())

Original Records : 119390
After Removing Duplicates : 87396


# Sort the Dataset

Sort the bookings based on Average Daily Rate (ADR) in descending order.

In [27]:
df.orderBy(
    col("adr").desc()
).show(10)

+------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+---------+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+---------------+------+---------------------------+-------------------------+------------------+-----------------------+
|       hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|     meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|  customer_type|   adr|required_car_par

#Display Distinct Countries

Find all unique countries from where bookings were made.

In [28]:
df.select(
    "country"
).distinct().show()

+-------+
|country|
+-------+
|    POL|
|    LVA|
|    ZMB|
|    JAM|
|    BRA|
|    ARM|
|    MOZ|
|    CUB|
|    JOR|
|    FRA|
|    URY|
|    GIB|
|    ETH|
|     CN|
|    ITA|
|    UKR|
|    CMR|
|    GHA|
|    SEN|
|    COM|
+-------+
only showing top 20 rows


#  Count Bookings by Hotel Type

Calculate the number of bookings for each hotel type.

In [29]:
df.groupBy(
    "hotel"
).count().show()

+------------+-----+
|       hotel|count|
+------------+-----+
|  City Hotel|79330|
|Resort Hotel|40060|
+------------+-----+



#Calculate Average Room Price

Calculate the average daily rate (ADR) for each hotel type.

In [30]:
df.groupBy(
    "hotel"
).agg(
    round(avg("adr"),2).alias("Average Room Price")
).show()

+------------+------------------+
|       hotel|Average Room Price|
+------------+------------------+
|  City Hotel|             105.3|
|Resort Hotel|             94.95|
+------------+------------------+



# Filter High Revenue Bookings

Display bookings where the Average Daily Rate (ADR) is greater than 200.

In [31]:
high_price = df.filter(
    col("adr") > 200
)

high_price.show(10)

+------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+------+---------------------------+-------------------------+------------------+-----------------------+
|       hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type|   adr|required_car_parking_spaces|to

# Performance Optimization and ETL Pipeline

In this section, we will explore Spark performance concepts such as Wide Transformations, Shuffle, Predicate Pushdown, and compare CSV and Parquet file formats. Finally, we will build a complete ETL pipeline.

# Perform a Wide Transformation using groupBy()

A Wide Transformation requires data to be shuffled across different partitions. Operations such as `groupBy()` and `join()` are examples of wide transformations because Spark redistributes data before processing.

In [33]:
# Count bookings by hotel type

hotel_count = df.groupBy("hotel").count()

hotel_count.show()

+------------+-----+
|       hotel|count|
+------------+-----+
|  City Hotel|79330|
|Resort Hotel|40060|
+------------+-----+



#  Display the Execution Plan

Display the logical and physical execution plans to observe how Spark processes the groupBy() operation.

In [34]:
hotel_count.explain(True)

== Parsed Logical Plan ==
'Aggregate ['hotel], ['hotel, 'count(1) AS count#3984]
+- Relation [hotel#17,is_canceled#18,lead_time#19,arrival_date_year#20,arrival_date_month#21,arrival_date_week_number#22,arrival_date_day_of_month#23,stays_in_weekend_nights#24,stays_in_week_nights#25,adults#26,children#27,babies#28,meal#29,country#30,market_segment#31,distribution_channel#32,is_repeated_guest#33,previous_cancellations#34,previous_bookings_not_canceled#35,reserved_room_type#36,assigned_room_type#37,booking_changes#38,deposit_type#39,agent#40,company#41,... 7 more fields] csv

== Analyzed Logical Plan ==
hotel: string, count: bigint
Aggregate [hotel#17], [hotel#17, count(1) AS count#3984L]
+- Relation [hotel#17,is_canceled#18,lead_time#19,arrival_date_year#20,arrival_date_month#21,arrival_date_week_number#22,arrival_date_day_of_month#23,stays_in_weekend_nights#24,stays_in_week_nights#25,adults#26,children#27,babies#28,meal#29,country#30,market_segment#31,distribution_channel#32,is_repeated_

#Understand Shuffle

A shuffle occurs when Spark redistributes data across partitions. Wide transformations such as groupBy(), join(), and distinct() trigger shuffle operations.

Shuffle is one of the most expensive operations because it involves:
- Data movement across executors
- Disk I/O
- Network communication

Reducing unnecessary shuffle improves Spark performance.

# Perform Aggregation

Calculate the average room price (ADR) for each hotel type.

In [35]:
df.groupBy("hotel") \
  .agg(
      round(avg("adr"),2).alias("Average ADR")
  ) \
  .show()

+------------+-----------+
|       hotel|Average ADR|
+------------+-----------+
|  City Hotel|      105.3|
|Resort Hotel|      94.95|
+------------+-----------+



# Demonstrate Predicate Pushdown

Predicate Pushdown allows Spark to apply filter conditions while reading the data instead of filtering after loading it completely. This reduces the amount of data read from storage.

In [36]:
filtered_df = df.filter(col("adr") > 150)

filtered_df.show(10)

+------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+------+---------------------------+-------------------------+------------------+-----------------------+
|       hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type|   adr|required_car_parking_spaces|to

# Display the Optimized Query Plan

Observe how Spark optimizes the filter operation.

In [37]:
filtered_df.explain(True)

== Parsed Logical Plan ==
'Filter '`>`('adr, 150)
+- Relation [hotel#17,is_canceled#18,lead_time#19,arrival_date_year#20,arrival_date_month#21,arrival_date_week_number#22,arrival_date_day_of_month#23,stays_in_weekend_nights#24,stays_in_week_nights#25,adults#26,children#27,babies#28,meal#29,country#30,market_segment#31,distribution_channel#32,is_repeated_guest#33,previous_cancellations#34,previous_bookings_not_canceled#35,reserved_room_type#36,assigned_room_type#37,booking_changes#38,deposit_type#39,agent#40,company#41,... 7 more fields] csv

== Analyzed Logical Plan ==
hotel: string, is_canceled: int, lead_time: int, arrival_date_year: int, arrival_date_month: string, arrival_date_week_number: int, arrival_date_day_of_month: int, stays_in_weekend_nights: int, stays_in_week_nights: int, adults: int, children: string, babies: int, meal: string, country: string, market_segment: string, distribution_channel: string, is_repeated_guest: int, previous_cancellations: int, previous_bookings_not

#Save the Processed Data as CSV

Write the filtered DataFrame into CSV format.

In [38]:
filtered_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("HotelBookings_CSV")

# Save the Processed Data as Parquet

Write the filtered DataFrame into Parquet format.

Parquet is a columnar storage format that provides better compression and faster analytical queries.

In [39]:
filtered_df.write \
    .mode("overwrite") \
    .parquet("HotelBookings_Parquet")

# Read the Parquet File

Read the Parquet file that was created in the previous step.

In [40]:
parquet_df = spark.read.parquet("HotelBookings_Parquet")

parquet_df.show(5)

+------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+------+---------------------------+-------------------------+------------------+-----------------------+
|       hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type|   adr|required_car_parking_spaces|to

#Compare CSV and Parquet Read Performance

Measure the approximate read time for both CSV and Parquet formats.

In [42]:
import time
import builtins

start = time.time()

csv_df = spark.read.csv(
    "hotel_bookings.csv",
    header=True,
    inferSchema=True
)

csv_df.count()

end = time.time()

print("CSV Read Time:", builtins.round(end - start, 2), "seconds")

CSV Read Time: 1.93 seconds


In [43]:
import time
import builtins

start = time.time()

parquet_df = spark.read.parquet("HotelBookings_Parquet")

parquet_df.count()

end = time.time()

print("Parquet Read Time:", builtins.round(end - start, 2), "seconds")

Parquet Read Time: 0.44 seconds


#Build an ETL Pipeline

Perform an end-to-end ETL pipeline:

- Read data
- Transform data
- Filter records
- Create new columns
- Write processed data

In [44]:
pipeline_df = spark.read.csv(
    "hotel_bookings.csv",
    header=True,
    inferSchema=True
)

pipeline_df = pipeline_df.filter(
    col("is_canceled") == 0
)

pipeline_df = pipeline_df.withColumn(
    "Total_Guests",
    col("adults") +
    col("children") +
    col("babies")
)

pipeline_df = pipeline_df.select(
    "hotel",
    "country",
    "arrival_date_year",
    "adr",
    "Total_Guests"
)

pipeline_df.show(10)

+------------+-------+-----------------+-----+------------+
|       hotel|country|arrival_date_year|  adr|Total_Guests|
+------------+-------+-----------------+-----+------------+
|Resort Hotel|    PRT|             2015|  0.0|           2|
|Resort Hotel|    PRT|             2015|  0.0|           2|
|Resort Hotel|    GBR|             2015| 75.0|           1|
|Resort Hotel|    GBR|             2015| 75.0|           1|
|Resort Hotel|    GBR|             2015| 98.0|           2|
|Resort Hotel|    GBR|             2015| 98.0|           2|
|Resort Hotel|    PRT|             2015|107.0|           2|
|Resort Hotel|    PRT|             2015|103.0|           2|
|Resort Hotel|    PRT|             2015|145.0|           2|
|Resort Hotel|    USA|             2015| 97.0|           2|
+------------+-------+-----------------+-----+------------+
only showing top 10 rows


#  Save the ETL Output

Save the transformed dataset as a Parquet file.

In [45]:
pipeline_df.write \
    .mode("overwrite") \
    .parquet("Processed_Hotel_Bookings")

#Display the Final Processed Dataset

In [46]:
processed_df = spark.read.parquet(
    "Processed_Hotel_Bookings"
)

processed_df.show(10)

+----------+-------+-----------------+---+------------+
|     hotel|country|arrival_date_year|adr|Total_Guests|
+----------+-------+-----------------+---+------------+
|City Hotel|    PRT|             2015|0.0|           2|
|City Hotel|    PRT|             2015|0.0|           1|
|City Hotel|    PRT|             2015|0.0|           1|
|City Hotel|    PRT|             2015|0.0|           1|
|City Hotel|    PRT|             2015|0.0|           1|
|City Hotel|    PRT|             2015|0.0|           1|
|City Hotel|    PRT|             2015|0.0|           2|
|City Hotel|    PRT|             2015|0.0|           2|
|City Hotel|    PRT|             2015|0.0|           1|
|City Hotel|    PRT|             2015|0.0|           1|
+----------+-------+-----------------+---+------------+
only showing top 10 rows


# Best Practices for Large Datasets

When working with large datasets in Spark:

- Use `show()` instead of `collect()` whenever possible.
- Filter data early to reduce processing.
- Prefer Parquet over CSV for analytics.
- Avoid unnecessary shuffle operations.
- Cache DataFrames only when reused multiple times.
- Select only the required columns.
- Handle null values before analysis.
- Use DataFrame APIs instead of Python loops.

# CSV vs Parquet Comparison

| Feature | CSV | Parquet |
|---------|------|----------|
| Storage Format | Row-Based | Column-Based |
| Compression | No | Yes |
| File Size | Larger | Smaller |
| Read Speed | Slower | Faster |
| Query Performance | Lower | Higher |
| Best Use | Data Exchange | Big Data Analytics |

For analytical workloads, Parquet is generally preferred because it stores data efficiently and supports optimized query execution.

# Assignment Summary

## Concepts Covered

- Spark Architecture
- Driver, Cluster Manager and Executors
- Spark Execution Modes
- Lazy Evaluation
- DAG (Directed Acyclic Graph)
- Reading CSV Files
- Schema Inference
- DataFrame Transformations
- Filtering and Selection
- Renaming Columns
- Casting Data Types
- Creating New Columns
- Handling Missing Values
- Removing Duplicates
- Wide Transformations
- Shuffle
- Predicate Pushdown
- CSV vs Parquet
- ETL Pipeline
- Saving Data
- Spark Performance Best Practices

## Conclusion

This assignment demonstrated the complete Spark data processing workflow using the Hotel Booking Demand dataset. We explored Spark architecture, performed DataFrame transformations, handled missing values, optimized processing through Spark's execution model, compared CSV and Parquet file formats, and built a complete ETL pipeline. These techniques are widely used in real-world Data Engineering projects for efficient processing of large-scale datasets.